## Data Evaluation
---

In [ ]:
"""
Set up the Python path so that the package `models.classification` is importable,
then pull in standard data-science libraries (pandas, numpy, matplotlib, seaborn) and the
project's utility functions and settings constants.
"""

import sys
from pathlib import Path

# Resolve repo + classification package paths regardless of notebook launch cwd.
CLASSIFICATION_DIR = Path.cwd()
if not (CLASSIFICATION_DIR / "settings.py").exists():
    CLASSIFICATION_DIR = Path.cwd() / "models" / "classification"
ROOT = CLASSIFICATION_DIR.parent.parent
for path in (CLASSIFICATION_DIR, ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.colors import Normalize

import importlib
import settings
importlib.reload(settings)
from _helpers import load_inputs_df
from models.classification.scores import (
    calc_normalized_deltas,
    cara_score,
    linex_score,
    naive_score,
    score_df,
)
from settings import (
    METRICS_CSV,
    INPUTS_CSV,
    C_TARGET_COLS,
    C_TARGET_LABELS,
    C_TARGET_LABEL,
    C_TARGET_COL,
    C_TARGET_FUNC,
    C_TARGET_ALPHA,
    TARGET_T_DELTA,
    DEFAULT_T_START,
    DEFAULT_T_END,
    PAPER_T_START,
    PAPER_T_END,
    PAPER_T_DELTA,
    PSNR_COL,
    CLIP_COL,
)


def _grid_levels(series):
    return sorted(series.astype(float).unique())

def _mean_pivot(sub, metric, row="t_start", col="t_end"):
    """Mean metric on the full rowxcol grid (no missing cells)."""
    rows, cols = _grid_levels(sub[row]), _grid_levels(sub[col])
    pivot = (
        sub.groupby([row, col], observed=True)[metric]
        .mean()
        .unstack(col)
    )
    pivot.index = pivot.index.astype(float)
    pivot.columns = pivot.columns.astype(float)
    return pivot.reindex(index=rows, columns=cols)

def _as_axes2d(axes):
    return np.atleast_2d(np.asarray(axes, dtype=object))

def _as_axes1d(axes):
    arr = np.atleast_1d(np.asarray(axes, dtype=object))
    return arr if arr.ndim == 1 else arr.ravel()

PLOT_LABELS = {**dict(C_TARGET_LABELS), C_TARGET_COL: C_TARGET_LABEL}


In [ ]:
"""
Load the raw metrics CSV into a DataFrame, compute the active target metric column in
memory for analysis, zero-pad sample IDs for consistent string representation, drop rows
missing any tracked metric, and cast the diffusion-parameter columns to ordered categoricals
so downstream plots and groupbys respect their natural sort order.

To persist computed metrics to METRICS_CSV, run add_data.ipynb instead.
"""

# Load metrics data from csv
df = pd.read_csv(METRICS_CSV)
print(f"Loaded: {METRICS_CSV}")

# Compute active metric column for evaluation (saved via add_data.ipynb)
df[C_TARGET_COL] = C_TARGET_FUNC(df)
df["sample_id"] = df["sample_id"].astype(str).str.zfill(12)
df["sample_id"] = df["sample_id"].astype(int)

na_mask = df[list(C_TARGET_COLS)].notna().all(axis=1)
print(f"Dropped {(~na_mask).sum()} rows")
df = df[na_mask]

# Cast parameter columns to categorical so that plots order them correctly
for col in ["t_start", "t_end", "t_delta"]:
    df[col] = pd.Categorical(df[col], categories=sorted(df[col].unique()), ordered=True)

t_delta_vals = df["t_delta"].cat.categories.tolist()
print(f"t_delta values: {t_delta_vals}")
print(f"Shape: {df.shape}")
df.describe()


### 1  Score surfaces over normalized metric deltas


In [ ]:
"""
Plot the score surfaces phi(delta) on the normalized-delta grid that
scores.calc_normalized_deltas actually produces, delta in [-1, 1]^2. delta = 0 is the
baseline edit at (DEFAULT_T_START, DEFAULT_T_END); delta_i > 0 improves metric i over it.

The white contour is the phi = 0 indifference curve — how much regression on one metric a
given gain on the other buys back. naive_score's is a straight line (the metrics are
perfectly substitutable), while cara_score and linex_score bow away from the origin: a
severe regression cannot be offset by gains elsewhere. That bow is why LINEX is the active
score. All three pass through phi(0) = 0, so the baseline row always scores exactly 0.
"""

GRID_MIN, GRID_MAX, GRID_SAMPLES = -1.0, 1.0, 161
ALPHA = C_TARGET_ALPHA

delta_axis = np.linspace(GRID_MIN, GRID_MAX, GRID_SAMPLES)
psnr_grid, clip_grid = np.meshgrid(delta_axis, delta_axis)
deltas = torch.stack(  # (G, G, 2), i.e. (..., N, C) with the metric axis last
    [
        torch.as_tensor(psnr_grid, dtype=torch.float64),
        torch.as_tensor(clip_grid, dtype=torch.float64),
    ],
    dim=-1,
)

naive_surface = naive_score(deltas).numpy()
cara_surface = cara_score(deltas, alpha=ALPHA).numpy()
linex_surface = linex_score(deltas, alpha=ALPHA).numpy()

surfaces = {
    "naive_score": naive_surface,
    f"cara_score ($\\alpha={ALPHA:g}$)": cara_surface,
    f"linex_score ($\\alpha={ALPHA:g}$) — active": linex_surface,
}
delta_psnr_label = f"$\\Delta$ {C_TARGET_LABELS[PSNR_COL]} (normalized)"
delta_clip_label = f"$\\Delta$ {C_TARGET_LABELS[CLIP_COL]} (normalized)"

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, (name, surface) in zip(axes, surfaces.items()):
    color_mesh = ax.contourf(psnr_grid, clip_grid, surface, levels=40, cmap="viridis")
    ax.contour(psnr_grid, clip_grid, surface, levels=40, colors="k", linewidths=0.2, alpha=0.35)
    ax.contour(psnr_grid, clip_grid, surface, levels=[0.0], colors="w", linewidths=1.4)
    fig.colorbar(color_mesh, ax=ax, label="$\\varphi(\\Delta)$")
    ax.scatter([0], [0], c="white", edgecolors="black", s=80, zorder=5, label="baseline $\\Delta=0$")
    ax.axhline(0, color="#666", lw=0.6, linestyle="--")
    ax.axvline(0, color="#666", lw=0.6, linestyle="--")
    ax.set(xlim=(GRID_MIN, GRID_MAX), ylim=(GRID_MIN, GRID_MAX),
           xlabel=delta_psnr_label, ylabel=delta_clip_label, title=name)
    ax.set_box_aspect(1)
    ax.legend(loc="upper left", fontsize=8)
fig.suptitle(
    "Score surfaces on normalized deltas (white = $\\varphi=0$ baseline-indifference curve)",
    fontsize=13,
)
plt.tight_layout()
plt.show()

score_min, score_max = float(linex_surface.min()), float(linex_surface.max())
score_padding = 0.05 * max(score_max - score_min, 1e-9)
z_axis_limits = (score_min - score_padding, score_max + score_padding)
isometric_elevation = np.degrees(np.arctan(1 / np.sqrt(2)))
plot_title = f"{C_TARGET_COL} ($\\alpha={ALPHA:g}$)"

fig = plt.figure(figsize=(8, 8))
for panel_index, (azimuth, view_label) in enumerate([(45, "NE"), (135, "NW"), (225, "SW"), (315, "SE")], 1):
    ax = fig.add_subplot(2, 2, panel_index, projection="3d")
    ax.plot_surface(psnr_grid, clip_grid, linex_surface, cmap="viridis", linewidth=0, antialiased=True, alpha=0.9)
    ax.scatter([0], [0], [0.0], c="white", edgecolors="black", s=80, depthshade=False)
    ax.set(xlim=(GRID_MIN, GRID_MAX), ylim=(GRID_MIN, GRID_MAX), zlim=z_axis_limits,
           xlabel="$\\Delta$PSNR", ylabel="$\\Delta$CLIP", zlabel=C_TARGET_COL)
    ax.set_box_aspect((1, 1, 1))
    ax.view_init(elev=isometric_elevation, azim=azimuth)
fig.suptitle(plot_title, fontsize=14, y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


### 2  Metric Distributions

In [ ]:
"""
Extend the metric distribution plot to include the active computed score column alongside
PSNR and CLIP similarity, showing how the derived score is distributed relative to the
two raw signals it is based on.
"""

fig, axes = plt.subplots(1, len(PLOT_LABELS), figsize=(5 * len(PLOT_LABELS), 4), squeeze=False)
for ax, m in zip(_as_axes1d(axes), PLOT_LABELS):
    ax.hist(df[m], color="silver", bins=60, edgecolor="none", alpha=0.8)
    ax.axvline(df[m].mean(), color="#cc0000", lw=1.0, label=f"mean={df[m].mean():.3f}")
    ax.axvline(df[m].median(), color="#cc0000", lw=1.0, linestyle="--", label=f"median={df[m].median():.3f}")
    ax.set_xlabel(PLOT_LABELS[m])
    ax.set_ylabel("Count")
    ax.set_title(PLOT_LABELS[m])
    ax.legend(fontsize=9)
fig.suptitle(f"Metric distributions for {C_TARGET_COL} on {METRICS_CSV.stem}", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


### 3  Effect of `t_start` and `t_end` on Metrics

Within each `t_delta` slice look at how the marginal means of `t_start`, `t_end` vary.

In [ ]:
"""
For each combination of t_delta level and tracked metric, compute the marginal mean ±
95% confidence interval as t_start and t_end each vary independently, then plot them as
overlaid error-bar lines. This reveals whether either parameter has a consistent
directional effect on output quality regardless of the other parameter's value.
"""

n_td = len(t_delta_vals)
fig, axes = plt.subplots(n_td, len(PLOT_LABELS), figsize=(6 * len(PLOT_LABELS), 5 * n_td),
                         sharey="col", squeeze=False)
colors = {"t_start": "#1f77b4", "t_end": "#ff7f0e"}

for row_idx, td in enumerate(t_delta_vals):
    sub = df[df["t_delta"] == td]
    for col_idx, m in enumerate(PLOT_LABELS):
        ax = axes[row_idx, col_idx]
        for param in ["t_start", "t_end"]:
            grp = sub.groupby(param, observed=True)[m].agg(["mean", "sem"])
            x = grp.index.astype(float)
            ax.errorbar(
                x, grp["mean"], yerr=1.96 * grp["sem"],
                marker="o", label=param, color=colors[param],
                capsize=4, linewidth=2,
            )
        ax.set_xlabel("Parameter value")
        ax.set_ylabel("Metric value" if col_idx == 0 else "")
        ax.set_title(f"{PLOT_LABELS[m]}, $\\delta={td}$")
        ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2g"))
        if col_idx == len(PLOT_LABELS) - 1:
            ax.legend(fontsize=9)

fig.suptitle(f"Marginal metric mean for {C_TARGET_COL} on {METRICS_CSV.stem}", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
"""
Complement the marginal line plots by showing the full metric distribution (median, IQR,
and outliers) at each discrete t_start and t_end level for both t_delta slices. Box plots
expose skew and variance across the 700 samples per level that are invisible in mean-only
plots.
"""

for td in t_delta_vals:
    sub = df[df["t_delta"] == td]
    fig, axes = plt.subplots(2, len(PLOT_LABELS), figsize=(6 * len(PLOT_LABELS), 8), sharey="col", squeeze=False)
    for col_idx, m in enumerate(PLOT_LABELS):
        for row_idx, param in enumerate(["t_start", "t_end"]):
            ax = axes[row_idx, col_idx]
            order = [str(v) for v in sorted(sub[param].cat.categories)]
            sns.boxplot(
                data=sub, x=param, y=m, order=order,
                ax=ax, color=colors[param], fliersize=1, linewidth=0.8,
            )
            ax.set_xlabel(param)
            ax.set_ylabel("Metric value" if col_idx == 0 else "")
            ax.set_title(f"{PLOT_LABELS[m]}" if row_idx == 0 else "")
    fig.suptitle(f"Distributions by t_start, t_end for {C_TARGET_COL} on {METRICS_CSV.stem}  (t_delta={td})", fontsize=13)
    plt.tight_layout()
    plt.show()


### 4  Heatmaps of `t_start` by `t_end`

In [ ]:
"""
Show the joint mean of each metric over the full grid of (t_start, t_end) combinations,
separately for each t_delta slice. Values are normalized to [0, 1] across both panels so
the color scale is directly comparable, making it easy to identify parameter regions that
are consistently strong or weak.
"""

for m in PLOT_LABELS:
    n_td = len(t_delta_vals)
    fig, axes = plt.subplots(1, n_td, figsize=(7 * n_td, 5), squeeze=False)
    axes = _as_axes1d(axes)

    pivots = {}
    for td in t_delta_vals:
        sub = df[df["t_delta"] == td]
        pivots[td] = _mean_pivot(sub, m, row="t_start", col="t_end")

    all_vals = np.concatenate([p.values.ravel() for p in pivots.values()])
    gmin, gmax = all_vals.min(), all_vals.max()
    denom = gmax - gmin if gmax > gmin else 1.0

    for ax, td in zip(axes, t_delta_vals):
        norm_pivot = (pivots[td] - gmin) / denom
        sns.heatmap(
            norm_pivot, ax=ax, annot=True, fmt=".3f",
            cmap="viridis", vmin=0, vmax=1,
            linewidths=0.4, linecolor="#e0e0e0",
            cbar_kws={"label": f"Normalized {PLOT_LABELS[m]}"},
            annot_kws={"size": 8},
        )
        ax.set_title(f"$\\delta$ = {td}")
        ax.set_xlabel("t_end")
        ax.set_ylabel("t_start")
        ax.invert_yaxis()
    fig.suptitle(f"Normalized mean {m} for {C_TARGET_COL} on {METRICS_CSV.stem}", fontsize=14)
    plt.tight_layout()
    plt.show()


In [ ]:
"""
Re-render the (t_start, t_end) mean-metric grids as 3D bar charts to give a spatial
intuition for the quality landscape: tall bars indicate high-performing parameter
combinations, and the viridis color gradient maps to the same normalized scale as the
2D heatmaps above.
"""

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

for m in PLOT_LABELS:
    n_td = len(t_delta_vals)
    fig = plt.figure(figsize=(7 * n_td, 5))

    pivots = {}
    for td in t_delta_vals:
        sub = df[df["t_delta"] == td]
        pivots[td] = _mean_pivot(sub, m, row="t_start", col="t_end")

    all_vals = np.concatenate([p.values.ravel() for p in pivots.values()])
    gmin, gmax = all_vals.min(), all_vals.max()
    denom = gmax - gmin if gmax > gmin else 1.0

    for i, td in enumerate(t_delta_vals):
        ax = fig.add_subplot(1, n_td, i + 1, projection="3d")
        norm_pivot = (pivots[td] - gmin) / denom

        xs = norm_pivot.index.values    # t_start
        ys = norm_pivot.columns.values  # t_end
        n_x, n_y = len(xs), len(ys)

        xpos = np.repeat(np.arange(n_x), n_y)
        ypos = np.tile(np.arange(n_y), n_x)
        zpos = np.zeros(n_x * n_y)
        dz   = norm_pivot.values.ravel()

        bar_colors = plt.cm.viridis(dz)
        ax.bar3d(xpos, ypos, zpos, dx=0.8, dy=0.8, dz=dz,
                 color=bar_colors, shade=True, alpha=0.85)

        ax.set_xticks(np.arange(n_x) + 0.4)
        ax.set_xticklabels([f"{v:.1g}" for v in xs], fontsize=7, rotation=45)
        ax.set_yticks(np.arange(n_y) + 0.4)
        ax.set_yticklabels([f"{v:.1g}" for v in ys], fontsize=7)
        ax.set_xlabel("t_start", labelpad=8)
        ax.set_ylabel("t_end",   labelpad=8)
        ax.set_zlim(0, 1)
        ax.view_init(elev=30, azim=225)

    fig.suptitle(
        f"Normalized 3D mean {PLOT_LABELS[m].lower()} for\n{C_TARGET_COL} on {METRICS_CSV.stem}",
        fontsize=14, y=1.02,
    )
    plt.tight_layout()
    plt.show()


### 5  Multimodality of `TARGET_COLUMN` over `(t_start, t_end)`

Averaging can hide structure: the **aggregate mean** surface often looks **unimodal** (one smooth peak), while **individual samples** frequently have **multiple local optima**. Plots are grouped by per-sample peak count.


In [ ]:
"""Multimodality helpers. Run mm-compute next to identify peak levels."""
from matplotlib.patches import Patch
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

_COLOR_MEAN, _COLOR_SAMPLE, _COLOR_UNI, _COLOR_MULTI = "#d62728", "#1f77b4", "#aec7e8", "#ff7f0e"
_PEAK_COLORS = plt.cm.viridis(np.linspace(0, 1, 5))

def _count_local_maxima_1d(arr):
    arr = np.asarray(arr, dtype=float).ravel()
    peaks = [i for i, v in enumerate(arr) if v > (arr[i-1] if i else -np.inf) and v > (arr[i+1] if i < len(arr)-1 else np.inf)]
    return len(peaks), peaks

def _count_local_maxima_2d(arr):
    arr = np.asarray(arr, dtype=float); ny, nx = arr.shape; peaks = []
    for yi in range(ny):
        for xi in range(nx):
            if np.isnan(arr[yi, xi]): continue
            if all(arr[yi+dy, xi+dx] < arr[yi, xi] for dy in (-1,0,1) for dx in (-1,0,1) if (dy,dx)!=(0,0) and 0<=yi+dy<ny and 0<=xi+dx<nx):
                peaks.append((yi, xi))
    return len(peaks), peaks

def _count_local_maxima(grid):
    grid = np.asarray(grid, dtype=float)
    return _count_local_maxima_1d(grid[:, 0] if grid.ndim == 2 and grid.shape[1] == 1 else grid) if grid.ndim == 1 or grid.shape[1] == 1 else _count_local_maxima_2d(grid)

def _sample_surface(sub, pivot_index, pivot_columns):
    return sub.groupby(["t_start", "t_end"], observed=True)[C_TARGET_COL].mean().unstack("t_end").reindex(index=pivot_index, columns=pivot_columns).to_numpy(dtype=float)

def _peak_coords(peaks, pivot_index, pivot_columns, grid_shape):
    return [(float(pivot_index[p]), 0) for p in peaks] if grid_shape[1] == 1 else [(float(pivot_index[p[0]]), float(pivot_columns[p[1]])) for p in peaks]

def _fmt_sample_id(sid): return str(int(sid)).zfill(12)

def _decorate_score_ax(ax, ylabel=False):
    ax.axhline(0, color="#666666", lw=0.8, linestyle="--", zorder=0)
    if ylabel: ax.set_ylabel(C_TARGET_COL, fontsize=10)

_D_TITLE = f"$\\delta={TARGET_T_DELTA}$"



In [ ]:
"""Build mm_by_td and identify distinct per-sample peak counts."""
mm_by_td = {}
for td in t_delta_vals:
    sub = df[df["t_delta"] == td]
    pivot = _mean_pivot(sub, C_TARGET_COL, row="t_start", col="t_end")
    grid = pivot.to_numpy(dtype=float); t_start_vals, t_end_vals = pivot.index.values, pivot.columns.values
    is_1d = grid.shape[1] == 1; n_global_peaks, global_peaks = _count_local_maxima(grid)
    sample_records = []
    for sid, grp in sub.groupby("sample_id"):
        sg = _sample_surface(grp, pivot.index, pivot.columns); n_peaks, peaks = _count_local_maxima(sg)
        yi, xi = np.unravel_index(int(np.nanargmax(sg)), sg.shape)
        sample_records.append({"sample_id": sid, "surface": sg, "n_peaks": n_peaks, "peaks": peaks, "argmax_ts": float(t_start_vals[yi]), "argmax_te": float(t_end_vals[xi])})
    counts = np.array([r["n_peaks"] for r in sample_records]); n_samples, n_multi = len(counts), int((counts > 1).sum()); n_uni, pct_multi = n_samples - n_multi, 100 * n_multi / n_samples
    peak_levels = sorted(int(k) for k in pd.Series(counts).unique())
    by_n_peaks = {k: [r for r in sample_records if r["n_peaks"] == k] for k in peak_levels}
    print(f"=== Multimodality: {C_TARGET_COL} (t_delta={td}) ===\nGrid: {grid.shape[0]} t_start x {grid.shape[1]} t_end levels\nMean surface: {n_global_peaks} local maximum — appears UNIMODAL")
    if global_peaks: print(f"  Mean peak at: {_peak_coords(global_peaks, pivot.index, pivot.columns, grid.shape)}")
    print(f"Per-sample: {n_uni} unimodal ({100-pct_multi:.1f}%), {n_multi} multimodal ({pct_multi:.1f}%)")
    for k, v in pd.Series(counts).value_counts().sort_index().items(): print(f"    {k} peak(s): {v} samples ({100*v/n_samples:.1f}%)")
    print()
    mm_by_td[td] = dict(pivot=pivot, grid=grid, t_start_vals=t_start_vals, t_end_vals=t_end_vals, is_1d=is_1d, n_global_peaks=n_global_peaks, global_peaks=global_peaks, sample_records=sample_records, by_n_peaks=by_n_peaks, peak_levels=peak_levels, n_samples=n_samples, n_multi=n_multi, n_uni=n_uni, pct_multi=pct_multi)

MM_PEAK_LEVELS = sorted({k for mm in mm_by_td.values() for k in mm["peak_levels"]})
print(f"Distinct peak counts: {MM_PEAK_LEVELS}")


In [ ]:
"""Mean (unimodal) vs per-sample curves, coloured by peak count."""
for td, mm in mm_by_td.items():
    pivot, grid, ts, is_1d, gpeaks, recs = mm["pivot"], mm["grid"], mm["t_start_vals"], mm["is_1d"], mm["global_peaks"], mm["sample_records"]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [2.2, 1]}); ax = axes[0]
    if is_1d:
        for rec in recs:
            clr = _PEAK_COLORS[(rec["n_peaks"] - 1) % len(_PEAK_COLORS)]
            ax.plot(ts, rec["surface"][:, 0], color=clr, alpha=0.2, linewidth=0.8, zorder=1)
        mean = grid[:, 0]
        ax.plot(ts, mean, color=_COLOR_MEAN, linewidth=1, label="Mean (aggregate)", zorder=5)
        for p in gpeaks: ax.scatter(ts[p], mean[p], s=200, marker="*", c=_COLOR_MEAN, edgecolors="white", linewidths=1.2, zorder=6, label="Mean peak" if p == gpeaks[0] else None)
        ax.set_xlabel("t_start"); _decorate_score_ax(ax, ylabel=True); ax.legend(loc="upper right", fontsize=8)
    else:
        sns.heatmap(pivot, ax=ax, cmap="Reds", linewidths=0.3, linecolor="#e8e8e8", cbar_kws={"label": f"Mean {C_TARGET_COL}"})
        for yi, xi in gpeaks: ax.scatter(xi + 0.5, yi + 0.5, s=200, marker="*", c=_COLOR_MEAN, edgecolors="white", linewidths=1.2, zorder=5)
        ax.set_xlabel("t_end"); ax.set_ylabel("t_start"); ax.invert_yaxis()
    ax.set_title(f"Score traces, {_D_TITLE}")
    ax = axes[1]
    levels, level_counts = mm["peak_levels"], [len(mm["by_n_peaks"][k]) for k in mm["peak_levels"]]
    bars = ax.bar([f"{k}\npeak{'s' if k != 1 else ''}" for k in levels], level_counts, color=[_PEAK_COLORS[(k - 1) % len(_PEAK_COLORS)] for k in levels], edgecolor="white", width=0.55)
    for bar, cnt in zip(bars, level_counts): ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + mm["n_samples"]*0.02, f"{cnt}\n({100*cnt/mm['n_samples']:.0f}%)", ha="center", va="bottom", fontsize=10)
    ax.set_ylabel("Sample count"); ax.set_ylim(0, mm["n_samples"] * 1.15); ax.set_title(f"Samples by peak count, {_D_TITLE}")
    fig.suptitle(f"Mean comparison for {C_TARGET_COL} on {METRICS_CSV.stem}", fontsize=14, y=1.02); plt.tight_layout(); plt.show()


In [ ]:
"""Where individual optima land vs the single mean peak."""

for td, mm in mm_by_td.items():
    pivot, ts, te, is_1d, gpeaks, recs = mm["pivot"], mm["t_start_vals"], mm["t_end_vals"], mm["is_1d"], mm["global_peaks"], mm["sample_records"]
    fig, ax = plt.subplots(figsize=(8, 5)); argmax_ts, argmax_te = np.array([r["argmax_ts"] for r in recs]), np.array([r["argmax_te"] for r in recs])
    if is_1d:
        ax.hist(argmax_ts, bins=len(ts), range=(ts.min()-0.05, ts.max()+0.05), color=_COLOR_MULTI, alpha=0.65, edgecolor="white", label="Per-sample argmax")
        for p in gpeaks: ax.axvline(ts[p], color=_COLOR_MEAN, lw=1.0, linestyle="--", label=f"Mean peak (t_start={ts[p]:.1f})")
        ax.set_xlabel("t_start of per-sample best pair"); ax.set_ylabel("Sample count"); ax.set_title(f"Individual optima scatter, {_D_TITLE}")
    else:
        sns.heatmap(pivot, ax=ax, cmap="Greys", alpha=0.35, linewidths=0.3, linecolor="#e0e0e0", cbar=False, annot=False)
        ax.scatter(argmax_te, argmax_ts, c=_COLOR_MULTI, alpha=0.35, s=18, edgecolors="none", label="Per-sample argmax")
        for yi, xi in gpeaks: ax.scatter(te[xi], ts[yi], s=250, marker="*", c=_COLOR_MEAN, edgecolors="white", linewidths=1.0, label="Mean peak", zorder=5)
        ax.set_xlabel("t_end"); ax.set_ylabel("t_start"); ax.invert_yaxis(); ax.set_title(f"Per-sample optima spread across grid, {_D_TITLE}")
    ax.legend(fontsize=9, loc="upper right")
    fig.suptitle(f"Individual best pair for {C_TARGET_COL} on {METRICS_CSV.stem}", fontsize=13)
    plt.tight_layout(); plt.show()


In [ ]:
"""Example samples for each distinct peak count identified above."""

def _plot_peak_examples(mm, n_peaks, n_show=3):
    recs = mm["by_n_peaks"].get(n_peaks, [])
    if not recs: return False
    pivot, grid, ts, is_1d, gpeaks = mm["pivot"], mm["grid"], mm["t_start_vals"], mm["is_1d"], mm["global_peaks"]
    picks = recs if len(recs) <= n_show else sorted(recs, key=lambda r: r["surface"].max(), reverse=True)[:n_show]
    fig, axes = plt.subplots(1, len(picks), figsize=(5 * len(picks), 4), squeeze=False)
    clr = _PEAK_COLORS[(n_peaks - 1) % len(_PEAK_COLORS)]
    for ax, rec in zip(axes[0], picks):
        if is_1d:
            curve = rec["surface"][:, 0]
            ax.plot(ts, curve, color=clr, linewidth=1.0, label=f"Sample ({n_peaks=})")
            ax.plot(ts, grid[:, 0], color=_COLOR_MEAN, linewidth=1.0, linestyle="--", label="Mean")
            for p in rec["peaks"]: ax.scatter(ts[p], curve[p], s=80, marker="o", c=clr, edgecolors="white", linewidths=0.8, zorder=4)
            for p in gpeaks: ax.scatter(ts[p], grid[p, 0], s=120, marker="*", c=_COLOR_MEAN, edgecolors="white", linewidths=0.8, zorder=5)
            _decorate_score_ax(ax); ax.set_xlabel("t_start", fontsize=10)
        else:
            sns.heatmap(pd.DataFrame(rec["surface"], index=pivot.index, columns=pivot.columns), ax=ax, cmap="Blues", cbar=False, linewidths=0.3, linecolor="#e0e0e0")
            for yi, xi in rec["peaks"]: ax.scatter(xi + 0.5, yi + 0.5, s=60, marker="o", c=clr, edgecolors="white")
            for yi, xi in gpeaks: ax.scatter(xi + 0.5, yi + 0.5, s=120, marker="*", c=_COLOR_MEAN, edgecolors="white")
            ax.invert_yaxis()
        ax.set_title(f"Sample {_fmt_sample_id(rec['sample_id'])}, {_D_TITLE}", fontsize=10); ax.legend(fontsize=8)
    label = "1 peak" if n_peaks == 1 else f"{n_peaks} peaks"
    if is_1d: fig.supylabel(C_TARGET_COL)
    fig.suptitle(f"Examples for {label} (n={len(recs)}) for {C_TARGET_COL} on {METRICS_CSV.stem}", fontsize=13, y=1.02)
    plt.tight_layout(); plt.show(); return True

for n_peaks in MM_PEAK_LEVELS:
    for td, mm in mm_by_td.items():
        _plot_peak_examples(mm, n_peaks)


In [ ]:
"""Mean surface from several viewing angles (aggregate appears smooth)."""

for td, mm in mm_by_td.items():
    grid, ts, te, is_1d, gpeaks = mm["grid"], mm["t_start_vals"], mm["t_end_vals"], mm["is_1d"], mm["global_peaks"]
    if is_1d:
        te0 = float(te[0]); span = max(0.05, 0.08 * (ts.max() - ts.min() or 1.0)); y_strip = np.array([te0 - span, te0 + span])
        X, Y = np.meshgrid(ts, y_strip); Z = np.tile(grid[:, 0], (len(y_strip), 1)); x_label, y_label = "t_start", "t_end"
        peak_xyz = lambda p: (ts[p], te0, grid[p, 0])
    else:
        X, Y = np.meshgrid(te, ts); Z = grid; x_label, y_label = "t_end", "t_start"
        peak_xyz = lambda ix: (te[ix[1]], ts[ix[0]], grid[ix[0], ix[1]])
    fig = plt.figure(figsize=(14, 11))
    for i, (elev, azim, label) in enumerate([(25, 45, "NE"), (25, 135, "NW"), (25, 225, "SW"), (60, 315, "high E")], start=1):
        ax = fig.add_subplot(2, 2, i, projection="3d"); ax.plot_surface(X, Y, Z, cmap="Reds", linewidth=0, antialiased=True, alpha=0.9)
        pts = [peak_xyz(p) for p in gpeaks] if is_1d else [peak_xyz(ix) for ix in gpeaks]
        for x, y, z in pts: ax.scatter(x, y, z, color=_COLOR_MEAN, s=100, depthshade=False)
        ax.set_xlabel(x_label, labelpad=6); ax.set_ylabel(y_label, labelpad=6); ax.set_zlabel(C_TARGET_COL, labelpad=6)
        ax.set_title(f"elev={elev}°, azim={azim}° ({label}), {_D_TITLE}", fontsize=10); ax.view_init(elev=elev, azim=azim)
    fig.suptitle(f"Aggregate mean surface — unimodal ({mm['n_global_peaks']} peak), {mm['pct_multi']:.0f}% of samples multimodal\nfor {C_TARGET_COL} on {METRICS_CSV.stem}  (t_delta={td})", fontsize=13, y=0.98)
    fig.legend(handles=[Patch(facecolor=_COLOR_MEAN, label=f"Mean peak ({mm['n_global_peaks']})"), Patch(facecolor=_COLOR_MULTI, label=f"Multimodal samples ({mm['n_multi']})")], loc="lower center", ncol=2, fontsize=10, frameon=False)
    plt.tight_layout(rect=[0, 0.04, 1, 0.95]); plt.show()


### 6  Metric Correlations

PSNR and CLIP similarity are independent signals; the combined score is a concave
scalarization of their normalized deltas.  A pairplot reveals how correlated they are in practice.

In [ ]:
"""
Draw a corner pairplot of all tracked metric columns on a random subset, colored by
t_delta, to measure how correlated PSNR, CLIP similarity, and the computed score are with
each other and whether t_delta creates distinct distributional clusters.
"""

N = min(5000, len(df))
sample = df.sample(n=N, random_state=42)
multi_td = len(t_delta_vals) > 1
cols = list(PLOT_LABELS) + (["t_delta"] if multi_td else [])
g = sns.pairplot(
    sample[cols],
    hue="t_delta" if multi_td else None,
    diag_kind="kde",
    plot_kws={"alpha": 0.3, "s": 10},
    corner=True,
)
g.fig.suptitle(f"Pairwise metric correlations for {C_TARGET_COL} on {METRICS_CSV.stem} ({N}-sample subset)", y=1.02, fontsize=13)
plt.show()

print("Pearson correlation matrix:")
df[list(PLOT_LABELS)].corr().round(3)


### 9  Top Samples by `TARGET_COLUMN`

In [ ]:
"""
Join the metrics DataFrame with the string-pair CSV to attach source and target prompts
to each row, then display the top-20 rows ranked by the active computed metric. This lets
us inspect which specific prompt edits are associated with the highest-scoring diffusion
parameters.
"""

strings = load_inputs_df()
# load_inputs_df keys sample_id as str; cell 2 cast the metrics frame's to int.
strings["sample_id"] = strings["sample_id"].astype(int)

merged = df.merge(strings[["sample_id", "source_prompt", "target_prompt"]], on="sample_id", how="left")

cols = ["sample_id", "source_prompt", "target_prompt", "t_start", "t_end", C_TARGET_COL]
top_samples = (
    merged.nlargest(20, C_TARGET_COL)[cols]
    .reset_index(drop=True)
)
top_samples.index += 1

pd.set_option("display.max_colwidth", 120)
display(top_samples)


In [ ]:
"""
Compare the paper's fixed default parameters (PAPER_T_START, PAPER_T_END) against the
per-sample oracle best: for each sample_id the row with the highest computed metric is
selected, and the two populations are compared side-by-side through a summary table,
overlaid metric-distribution histograms, and bar charts showing which t_start and t_end
values the oracle most frequently selects.

NOTE: this section brackets the paper's PAPER_T_START, while the score itself measures
deltas against DEFAULT_T_START (the nearest on-grid value). The two differ, so the
"default" column here is not the row that scores exactly 0.
"""

# Only consider rows with valid PSNR and CLIP scores
valid = df.dropna(subset=C_TARGET_COLS)

# Mask for rows at the default parameter pair
default_mask = (
    (valid["t_start"].astype(float) == PAPER_T_START) &
    (valid["t_end"].astype(float) == PAPER_T_END)
)
df_default = valid[default_mask]

# Best per sample_id, row with the highest C_TARGET_COL for each image
best_idx = valid.groupby("sample_id")[C_TARGET_COL].idxmax()
df_best = valid.loc[best_idx]

# --- Descriptive stats side-by-side ---
comparison = pd.DataFrame({
    f"default ({PAPER_T_START}, {PAPER_T_END})": df_default[list(C_TARGET_COLS)].mean(),
    "per-sample best": df_best[list(C_TARGET_COLS)].mean(),
}).T
comparison.columns = list(C_TARGET_COLS)
display(comparison.round(4))

# --- Metric distribution comparison ---
fig, axes = plt.subplots(1, len(C_TARGET_COLS), figsize=(5 * len(C_TARGET_COLS), 4), squeeze=False)
axes = _as_axes1d(axes)
colors_grp = {"default": "#5b9bd5", "best": "#ed7d31"}

for ax, m in zip(axes, C_TARGET_COLS):  # axes is 1d
    for label, sub, clr in [
        (f"default ({PAPER_T_START}, {PAPER_T_END})", df_default, colors_grp["default"]),
        ("per-sample best", df_best, colors_grp["best"]),
    ]:
        ax.hist(sub[m], bins=40, alpha=0.55, color=clr, label=label, edgecolor="none", density=True)
        ax.axvline(sub[m].mean(), color=clr, lw=2, linestyle="--")

    ax.set_xlabel(PLOT_LABELS[m])
    ax.set_ylabel("Density")
    ax.set_title(f"{PLOT_LABELS[m]}, $\\delta={TARGET_T_DELTA}$")
    ax.legend(fontsize=8)

fig.suptitle(
    f"Metric distributions for {C_TARGET_COL} on {METRICS_CSV.stem}: default vs per-sample best",
    fontsize=13,
)
plt.tight_layout()
plt.show()

# --- t_start / t_end distribution of per-sample best rows ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, param in zip(axes, ["t_start", "t_end"]):
    counts = df_best[param].astype(float).value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values, color="#ed7d31", edgecolor="none")
    ax.set_xlabel(param)
    ax.set_ylabel("Count")
    ax.set_title(f"Distribution of best {param}, $\\delta={TARGET_T_DELTA}$")
    ax.tick_params(axis="x", rotation=45)

fig.suptitle(
    f"Distributions t_start,t_end (per-sample best) for {C_TARGET_COL} on {METRICS_CSV.stem}",
    fontsize=13,
)
plt.tight_layout()
plt.show()


### 10  LINEX Score Analysis

For each `sample_id` group the **baseline** row is `(t_start=DEFAULT_T_START, t_end=DEFAULT_T_END)`.
Each metric is min-max scaled across that sample's candidate cells, then the baseline is
subtracted:

$$\Delta_i = \frac{s_i - s_i^0}{\max_T s_i - \min_T s_i} \in [-1, 1]$$

The score is $\varphi_{\text{LINEX}}(\Delta) = \tfrac{1}{2}\sum_i \left[\Delta_i + (1 - e^{-\alpha \Delta_i})/\alpha\right]$.
The baseline scores exactly $0$, positive means a net improvement over the baseline edit, and
a severe regression on one metric cannot be bought back by a large gain on the other.

Computed fresh via `scores.score_df(..., score_fn=linex_score)` regardless of `C_TARGET_FUNC`.


In [ ]:
"""
Analyse the LINEX score in detail. Each row is classified as net worse than the baseline
edit (phi<0), the baseline itself (phi=0), or a net improvement (phi>0). The section reports
category counts, plots the score distribution, shows score-bucket bar charts, renders
(t_start, t_end) heatmaps of improvement fraction and mean score, and draws scatter plots of
ΔPSNR vs ΔCLIP for a random sample, for each sample's best-scoring row, and on the
normalized-delta axes the score is actually defined on.
"""

from matplotlib.patches import Patch

# Always recompute directly — independent of C_TARGET_FUNC
df["_lx"] = score_df(df, *C_TARGET_COLS, score_fn=linex_score, alpha=C_TARGET_ALPHA)

# score_df reduces the metric axis away, so re-pack the same grids to also keep the
# per-metric normalized deltas Δ for the surface panel at the end of this cell.
_groups = list(df.groupby("sample_id", sort=True))
_values = torch.as_tensor(
    np.stack([g.loc[:, list(C_TARGET_COLS)].to_numpy(dtype=float) for _, g in _groups]),
    dtype=torch.float64,
)
_baseline_idx = torch.as_tensor(
    [
        int(np.flatnonzero(
            np.isclose(g["t_start"].to_numpy(dtype=float), DEFAULT_T_START)
            & np.isclose(g["t_end"].to_numpy(dtype=float), DEFAULT_T_END)
        )[0])
        for _, g in _groups
    ],
    dtype=torch.long,
)
_norm_deltas = calc_normalized_deltas(_values, _baseline_idx).numpy()  # (B, N, C)
for _delta_col, _metric_axis in (("_dn_psnr", 0), ("_dn_clip", 1)):
    df[_delta_col] = np.nan
    for (_, _group), _block in zip(_groups, _norm_deltas):
        df.loc[_group.index, _delta_col] = _block[:, _metric_axis]

# Isolate pairs that beat the baseline edit
improving     = df["_lx"] > 0
baseline_rows = (
    np.isclose(df["t_start"].astype(float), DEFAULT_T_START) &
    np.isclose(df["t_end"].astype(float), DEFAULT_T_END)
)

"""
Report general statistics about the data
"""

n_total     = len(df)
n_improving = int(improving.sum())
n_baseline  = int(baseline_rows.sum())
n_neither   = n_total - n_improving - n_baseline

print(f"Total rows      : {n_total:,}")
print(f"Baseline rows   : {n_baseline:,}  (phi=0 exactly, one per sample_id)")
print(f"Net better      : {n_improving:,}  ({100*n_improving/n_total:.1f}%,  phi>0)")
print(f"Net worse       : {n_neither:,}  ({100*n_neither/n_total:.1f}%,  phi<0)")
all_scores = df["_lx"]
imp_scores = df.loc[improving, "_lx"]
print(f"\nAll rows   phi — min:{all_scores.min():.3f}  mean:{all_scores.mean():.3f}  max:{all_scores.max():.3f}")
if n_improving:
    print(f"Net better phi — min:{imp_scores.min():.3f}  mean:{imp_scores.mean():.3f}  max:{imp_scores.max():.3f}")


"""
Category counts, histogram, score buckets
"""

cmap = plt.get_cmap("Greys")

bin_edges = [-float("inf"), -0.5, -0.1, 0.0, 0.1, 0.5, float("inf")]
bin_labels = ["<-0.5", "(-0.5,-0.1]", "(-0.1,0]", "(0,0.1]", "(0.1,0.5]", ">0.5"]
bkt_colors = [cmap(x) for x in np.linspace(0.2, 0.8, 6)]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
cat_labels = ["Net worse\n($\\varphi<0$)", "Baseline\n($\\varphi=0$)", "Net better\n($\\varphi>0$)"]
cat_counts  = [n_neither, n_baseline, n_improving]
bars = ax.bar(cat_labels, cat_counts, color=[cmap(x) for x in np.linspace(0.2, 0.8, 3)], edgecolor="none")
for bar, cnt in zip(bars, cat_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + n_total*0.005,
            f"{cnt:,}\n({100*cnt/n_total:.1f}%)", ha="center", va="bottom", fontsize=8)
ax.set_ylabel("Pair count")
ax.set_title("Score categories")

ax = axes[1]
# The score is signed and unbounded, so bin every row rather than only the improvements.
ax.hist(all_scores, bins=40, color="silver", edgecolor="none", alpha=0.85)
ax.axvline(0.0, color="k", lw=1.0, label="baseline ($\\varphi=0$)")
ax.axvline(all_scores.median(), color="#cc0000", lw=1.0, linestyle="-", label=f"median={all_scores.median():.2f}")
ax.axvline(all_scores.mean(), color="#cc0000", lw=1.0, linestyle="--", label=f"mean={all_scores.mean():.2f}")
ax.set_xlabel(C_TARGET_COL)
ax.set_ylabel("Pair count")
ax.legend(fontsize=8)
ax.set_title("Score distribution (0 = baseline)")

ax = axes[2]
bucket_counts = pd.cut(df["_lx"], bins=bin_edges, labels=bin_labels).value_counts().reindex(bin_labels)
b = ax.bar(bin_labels, bucket_counts.values, color=bkt_colors, edgecolor="none")
for bar, cnt in zip(b, bucket_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + n_total*0.003,
            f"{cnt:,}", ha="center", va="bottom", fontsize=8)
ax.set_xlabel("Score bucket (the baseline's exact 0 falls in $(-0.1,0]$)")
ax.set_ylabel("Pair count")
ax.set_title("Score buckets")

fig.suptitle(f"{C_TARGET_LABEL} for {C_TARGET_COL} on {METRICS_CSV.stem}", fontsize=14)
plt.tight_layout()
plt.show()

"""
Heatmaps
"""

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pivot_frac = df.groupby(["t_start", "t_end"], observed=True).apply(
    lambda g: (g["_lx"] > 0).mean(), include_groups=False
).unstack("t_end")
pivot_mean = _mean_pivot(df, "_lx", row="t_start", col="t_end")
pivot_frac = pivot_frac.reindex(index=pivot_mean.index, columns=pivot_mean.columns)

for ax, pivot, title, cmap_name, center in [
    (axes[0], pivot_frac, f"Fraction with $\\varphi>0$ over baseline, $\\delta={TARGET_T_DELTA}$", "viridis", None),
    (axes[1], pivot_mean, f"Mean {C_TARGET_COL}, $\\delta={TARGET_T_DELTA}$", "RdBu_r", 0),
]:
    pivot.index   = pivot.index.astype(float)
    pivot.columns = pivot.columns.astype(float)
    sns.heatmap(pivot, ax=ax, annot=True, fmt=".2f", cmap=cmap_name, center=center,
                linewidths=0.4, linecolor="#e0e0e0", cbar_kws={"label": title})
    ax.set_title(title)
    ax.set_xlabel("t_end")
    ax.set_ylabel("t_start")
    ax.invert_yaxis()

fig.suptitle(f"{C_TARGET_LABEL} heatmap for {C_TARGET_COL} on {METRICS_CSV.stem}", fontsize=13)
plt.tight_layout()
plt.show()

"""
Scatter Plots
"""

# Compute per-sample PSNR/CLIP deltas
_base = (
    df[baseline_rows][["sample_id", PSNR_COL, CLIP_COL]]
    .rename(columns={PSNR_COL: "_b_psnr", CLIP_COL: "_b_clip"})
)
_sc = df.merge(_base, on="sample_id", how="inner")
_sc["_dpsnr"] = _sc[PSNR_COL] - _sc["_b_psnr"]
_sc["_dclip"]  = _sc[CLIP_COL] - _sc["_b_clip"]
_sc = _sc[~(np.isclose(_sc["_dpsnr"], 0) & np.isclose(_sc["_dclip"], 0))]

# Best row per sample_id: highest LINEX score, ties broken by order
_sc_best = _sc.loc[_sc.groupby("sample_id")["_lx"].idxmax()]
_baseline_pair = f"$({DEFAULT_T_START:g}, {DEFAULT_T_END:g})$"

def _scatter_plot(ax, data, title, sample_size=None, color_imp="#1f77b4", color_no="#aec7e8"):
    if sample_size and len(data):
        data = data.sample(n=min(sample_size, len(data)), random_state=42)
    _imp = data["_lx"] > 0
    ax.scatter(data.loc[~_imp, "_dpsnr"], data.loc[~_imp, "_dclip"],
               c=color_no, alpha=0.5, s=8, edgecolors="none", label="$\\varphi\\leq0$")
    ax.scatter(data.loc[_imp,  "_dpsnr"], data.loc[_imp,  "_dclip"],
               c=color_imp, alpha=0.5,  s=8, edgecolors="none", label="$\\varphi>0$")
    ax.axhline(0, color="#666", lw=0.8, linestyle="--")
    ax.axvline(0, color="#666", lw=0.8, linestyle="--")
    ax.set_xlabel(f"$\\Delta$PSNR to baseline {_baseline_pair}")
    ax.set_ylabel(f"$\\Delta$CLIP-Edited to baseline {_baseline_pair}")
    ax.set_title(title)
    ax.legend(fontsize=9, markerscale=3)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
_scatter_plot(axes[0], _sc, f"5k samples from all pairs, $\\delta={TARGET_T_DELTA}$", sample_size=5000)
_scatter_plot(axes[1], _sc_best, f"Highest `{C_TARGET_COL}` pair by sample_id, $\\delta={TARGET_T_DELTA}$", color_imp="#ff7f0e", color_no="#ffd5a8")

# Third panel: the same cloud on the normalized-Δ axes the score is defined on, over the
# LINEX surface from section 1. The white curve is where φ crosses 0.
ax = axes[2]
_surf_axis = np.linspace(-1.0, 1.0, 161)
_surf_psnr, _surf_clip = np.meshgrid(_surf_axis, _surf_axis)
_surf = linex_score(
    torch.stack([
        torch.as_tensor(_surf_psnr, dtype=torch.float64),
        torch.as_tensor(_surf_clip, dtype=torch.float64),
    ], dim=-1),
    alpha=C_TARGET_ALPHA,
).numpy()
ax.contourf(_surf_psnr, _surf_clip, _surf, levels=30, cmap="viridis", alpha=0.35)
ax.contour(_surf_psnr, _surf_clip, _surf, levels=[0.0], colors="w", linewidths=1.4)
_cloud = df.sample(n=min(5000, len(df)), random_state=42)
_cloud_imp = _cloud["_lx"] > 0
ax.scatter(_cloud.loc[~_cloud_imp, "_dn_psnr"], _cloud.loc[~_cloud_imp, "_dn_clip"],
           c="#aec7e8", alpha=0.5, s=8, edgecolors="none", label="$\\varphi\\leq0$")
ax.scatter(_cloud.loc[_cloud_imp, "_dn_psnr"], _cloud.loc[_cloud_imp, "_dn_clip"],
           c="#1f77b4", alpha=0.5, s=8, edgecolors="none", label="$\\varphi>0$")
ax.set(xlim=(-1, 1), ylim=(-1, 1),
       xlabel="$\\Delta$PSNR (normalized)", ylabel="$\\Delta$CLIP-Edited (normalized)",
       title="Data cloud on the LINEX surface")
ax.set_box_aspect(1)
ax.legend(fontsize=9, markerscale=3, loc="upper left")

fig.suptitle(
    f"ΔPSNR by ΔCLIP for {C_TARGET_COL} on {METRICS_CSV.stem}",
    fontsize=13
)
fig.text(
    0.5, 0.01,
    rf"A pair $(t^*, t^{{**}})$ scores $\varphi_{{\mathrm{{LINEX}}}}(\Delta) = \tfrac{{1}}{{2}}\sum_i"
    rf" \left[\Delta_i + (1 - e^{{-\alpha \Delta_i}})/\alpha\right]$ on deltas normalized within its"
    rf" sample, relative to the baseline $({DEFAULT_T_START:g}, {DEFAULT_T_END:g})$;"
    rf" the baseline itself scores $0$. $\alpha={C_TARGET_ALPHA:g}$.",
    ha="center",
    va="bottom",
    fontsize=8,
    color="0.2"
)
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()

### 11  Improving vs Non-improving Images (best row per sample)

For each image the single best-scoring parameter row is selected, ranked by `linex_score`.
We then split images into those whose best row Pareto-improves over the baseline on **both**
raw metrics vs those that do not — a stricter cut than $\varphi>0$, which allows trading a
regression on one metric for a larger gain on the other.


In [ ]:
"""
Join the per-sample best-row table with the prompt-string CSV and split images into those
whose best parameter pair Pareto-improves over the baseline on both PSNR and CLIP versus
those that do not, then display the top-20 rows from each group ordered by LINEX score.
"""

if "_sc_best" not in globals() or _sc_best is None or _sc_best.empty:
    raise RuntimeError("Run the LINEX Score section first to build `_sc_best`.")

_strings = load_inputs_df()
# load_inputs_df keys sample_id as str; cell 2 cast the metrics frame's to int.
_strings["sample_id"] = _strings["sample_id"].astype(int)

_best_labeled = _sc_best.merge(
    _strings[["sample_id", "source_prompt", "target_prompt"]],
    on="sample_id", how="left",
)
_best_labeled["improving"] = (_best_labeled["_dpsnr"] > 0) & (_best_labeled["_dclip"] > 0)

_display_cols = [
    "sample_id", "source_prompt", "target_prompt",
    "t_start", "t_end", "_dpsnr", "_dclip", "_lx",
]

_improving_df    = _best_labeled[_best_labeled["improving"]][_display_cols].sort_values("_lx", ascending=False).reset_index(drop=True)
_notimproving_df = _best_labeled[~_best_labeled["improving"]][_display_cols].sort_values("_lx", ascending=True).reset_index(drop=True)

_improving_df.index    += 1
_notimproving_df.index += 1

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 20)

n_best = len(_best_labeled)
if n_best:
    print(f"Improving images   : {len(_improving_df):,}  ({100*len(_improving_df)/n_best:.1f}%)")
    print(f"Non-improving      : {len(_notimproving_df):,}  ({100*len(_notimproving_df)/n_best:.1f}%)")
else:
    print("No per-sample best rows found — check that baseline rows exist in the data.")

print(f"\n── Top improving images (ranked by LINEX score) ──")
display(_improving_df.head(20).rename(columns={
    "_dpsnr": "ΔPSNR", "_dclip": "ΔCLIP", "_lx": "linex_score"
}))

print(f"\n── Non-improving images (best row still doesn't beat baseline on both metrics) ──")
display(_notimproving_df.head(20).rename(columns={
    "_dpsnr": "ΔPSNR", "_dclip": "ΔCLIP", "_lx": "linex_score"
}))
